# Paper and supplementary figures

Every figure in `docs/paper/manuscript.tex` and `docs/paper/supplementary.tex`, one
section each: a **run** cell that generates the data, then a **plot** cell that
re-plots it and shows it inline.

Nothing here duplicates simulation or plotting logic. The run cells call the same
entry points the SLURM jobs call; the plot cells drive the manifest in
`package/paper_figures/build_figures.py`, so the figure numbering, the artifact
filenames and the results folders are read from that one place.

## How to use it

1. Run the **Setup** cell.
2. Run the **Status** cell to see which figures already have data.
3. Run any plot cell: it re-plots from the folder in `build_figures.RUNS` and
   displays the PNG.
4. To regenerate data, set `RUN_GENERATION = True` in Setup and run that figure's
   run cell. With `RUN_GENERATION = False` (the default) every run cell is a no-op,
   so "Run All" plots everything and starts no simulations.

The run cells are written for a workstation. Most of these jobs were produced on a
cluster (see the `submit_*.slurm` file named in each section, and
`package/supplementary_runs/README.md` for run counts and wall-clock). Anything
above a few hundred runs is not a sensible local job: supplementary Figures 7/8
alone are 196,608 runs.

`STAGE_TO_PAPER = True` also copies each plotted PNG to its `docs/paper/` name
(`Figure_<n>.png`, `supplementary_figs/Supp_Figure_<n>.png`). The `.tex` files are
never touched from this notebook: for that, use
`python -m package.paper_figures.build_figures`.

| Figure | Section | Generated by |
|---|---|---|
| 1 | Model structure diagram | not model output (`docs/paper/static_figs/`) |
| 2 | Calibration validation | `package/generating_data/calibration_gen.py` |
| 3 | Single policies | `package/analysis/vary_single_policy_gen.py` |
| 4 | Policy pairs | `package/analysis/endogenous_policy_intensity_pair_gen.py` |
| 5 | Low-intensity policy mixes | `package/analysis/low_policy_intensity_gen.py` |
| 6 | Consumer preferences | `package/generating_data/single_experiment_gen.py` |
| 7 | Used car market capacity | `package/generating_data/sen_vary_single_param_gen_second_hand_cars.py` |
| S1 | NPE posterior | `package/calibration/sbi_single_seed_gen.py` |
| S2-S4 | External data / NK landscape | not model output (`docs/paper/static_figs/`) |
| S5 | Simulated vs real cars | same run as Figure 2 |
| S6 | Local sensitivity, 10 panels | `package/supplementary_runs/fig06_local_sensitivity_gen.py` |
| S7, S8 | Sobol indices | `package/supplementary_runs/fig07_08_sobol_gen.py` |
| S9, S10 | BAU grid and elasticities | `package/supplementary_runs/fig09_10_bau_gen.py` |
| S11-S14 | Policy grids | `package/supplementary_runs/fig11_14_policy_grid_gen.py` |

## Setup

In [ ]:
import glob
import os
import subprocess
import sys

from IPython.display import Image, Markdown, display

# ---- flags ----------------------------------------------------------------
RUN_GENERATION = False   # True = run cells actually simulate. Read the cost note first.
STAGE_TO_PAPER = False   # True = also copy each plotted PNG into docs/paper/
DISPLAY_WIDTH = 950      # pixels; the PNGs themselves are 300 dpi

# ---- repo root ------------------------------------------------------------
REPO_ROOT = os.path.abspath(os.getcwd())
if not os.path.isdir(os.path.join(REPO_ROOT, "package")):
    raise RuntimeError(f"Run this notebook from the repo top level, not {REPO_ROOT}")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# build_figures holds the manifest: figure number -> label, results folder, artifact,
# plotting call. It also forces the Agg backend, so figures are displayed from the
# PNG each plotting function saves rather than from an inline canvas.
import package.paper_figures.build_figures as bf

FIG = {figure.key: figure for figure in bf.FIGURES}


def newest(prefix):
    """Newest results/<prefix>_<timestamp> folder. The gen scripts print the folder
    they made; several then discard it, so resolve it the way the slurm jobs do."""
    matches = [p for p in glob.glob(f"results/{prefix}_*") if os.path.isdir(p)]
    if not matches:
        raise FileNotFoundError(f"No results/{prefix}_* folder found")
    return max(matches, key=os.path.getmtime).replace("\\", "/")


def run_module(module, *args):
    """`python -m module args` in a subprocess, output streamed, same environment
    settings as the slurm jobs (Agg, unbuffered, one BLAS thread per worker)."""
    env = dict(
        os.environ,
        MPLBACKEND="Agg",
        PYTHONUNBUFFERED="1",
        OMP_NUM_THREADS="1",
        MKL_NUM_THREADS="1",
        OPENBLAS_NUM_THREADS="1",
    )
    command = [sys.executable, "-m", module, *(str(a) for a in args)]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=REPO_ROOT, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{module} exited with code {process.returncode}")


def guard(cost):
    """Gate for the run cells. Prints the cost note and returns RUN_GENERATION."""
    print(f"Cost: {cost}")
    if not RUN_GENERATION:
        print("RUN_GENERATION is False, so nothing is generated. Set it to True to run.")
    return RUN_GENERATION


def using(run_key):
    """Report which folder a figure will be plotted from."""
    value = bf.RUNS.get(run_key, "")
    print(f"RUNS['{run_key}'] = {value if value else '<empty>'}")


def plot(*keys, replot=True):
    """Re-plot the given figures from their RUNS folder and show them inline.

    replot=False reuses the PNG already in the results folder.
    """
    for key in keys:
        figure = FIG[key]
        folders = figure.folders()

        if not figure.is_static and not folders:
            print(f"Figure {key}: RUNS['{figure.run_key}'] is empty. Generate it first.")
            continue
        missing = [f for f in folders if not os.path.isdir(bf._abs(f))]
        if missing:
            print(f"Figure {key}: folder not found: {missing[0]}")
            continue

        if replot and not figure.is_static:
            print(f"Plotting via {figure.replot.__name__} ...", flush=True)
            try:
                argument = folders if figure.run_key == "local_sensitivity" else folders[0]
                figure.replot(argument)
            except Exception as error:
                # One bad figure must not stop the rest, same as build_figures.
                print(f"Figure {key} failed to plot: {type(error).__name__}: {error}")
                continue
            finally:
                bf.plt.close("all")

        source = figure.source_path()
        if not source or not os.path.isfile(source):
            print(f"Figure {key}: no PNG at {source}")
            continue

        if STAGE_TO_PAPER:
            bf.build_figures([figure], skip_plot=True)

        name = f"{'Supplementary ' if figure.supplementary else ''}Figure {figure.number}"
        display(Markdown(
            f"**{name}**: {figure.caption}  \n`{os.path.relpath(source, REPO_ROOT)}`"
        ))
        display(Image(filename=source, width=DISPLAY_WIDTH))


print("Repo root:", REPO_ROOT)
print("Figures in the manifest:", ", ".join(FIG))

## Status

What is ready to plot, what still needs a run. Same output as
`python -m package.paper_figures.build_figures --list`.

In [ ]:
bf.print_manifest()

---
# Manuscript figures

## Figure 1: model structure diagram

Not model output. It is a hand-drawn diagram, already sitting at
`docs/paper/Figure_1.png`; a replacement goes in `docs/paper/static_figs/Figure_1.png`.
There is nothing to run.

In [ ]:
plot("1")

## Figure 2: calibration validation dashboard, 2001-2023

`package/supplementary_runs/fig05_calibration_cars_gen.py` wraps
`package/generating_data/calibration_gen.py` against
`package/constants/base_params_calibration.json`. One run makes both this figure
(`calibration_fit.png`) and supplementary Figure 5 (`multi_seed_2d_scatter.png`),
so run it once here and skip supplementary Figure 5's run cell.

Slurm: `package/supplementary_runs/submit_fig05_calibration_cars_gen.slurm`
(64 cores, 2 h wall-clock cap).

In [ ]:
if guard("64 runs (1 parameter set x 64 seeds), duration_future=0"):
    from package.supplementary_runs.fig05_calibration_cars_gen import run_fig5

    bf.RUNS["calibration"] = run_fig5()

using("calibration")

In [ ]:
plot("2")

## Figure 3: key model outputs for single policies

`package/analysis/vary_single_policy_gen.py` sweeps five policies over 100
intensities each, on top of 64 calibrated controllers. Configuration (policy list,
repetitions, bounds file) lives in that script's `__main__` block, so it is invoked
as a module rather than reassembled here.

Slurm: `package/analysis/submit_vary_single_policy_gen.slurm` (64 cores, 3 day cap).
This is a cluster job, not a laptop job.

In [ ]:
if guard("64 calibration runs + 5 policies x 100 intensities x 64 seeds = 32,064 runs"):
    run_module("package.analysis.vary_single_policy_gen")
    bf.RUNS["single_policies"] = newest("vary_single_policy_gen")

using("single_policies")

In [ ]:
plot("3")

## Figure 4: policy pair emissions, utility and cost trade-offs

`package/analysis/endogenous_policy_intensity_pair_gen.py` runs Bayesian
optimisation to hit a 95% EV uptake target for every policy pair, and writes
`Data/base_params`, `Data/outcomes_BAU`, `Data/single_policy_outcomes` and
`Data/pairwise_outcomes` into one `results/endog_pair_<stamp>` folder. Figure 5
consumes that same folder.

Slurm: `package/analysis/submit_endogenous_policy_intensity_pair_gen.slurm`
(64 cores, 7 day cap, the longest of the analysis jobs).

In [ ]:
if guard("BO over 10 policy pairs + 5 single policies, 64 seeds per evaluation. Days on 64 cores."):
    run_module("package.analysis.endogenous_policy_intensity_pair_gen")
    bf.RUNS["policy_pairs"] = newest("endog_pair")

using("policy_pairs")

In [ ]:
plot("4")

## Figure 5: policy mix trajectories to 2050

`package/analysis/low_policy_intensity_gen.py` takes the pair-gen folder from
Figure 4, selects the pairs landing in a 0.94-0.96 EV uptake window, and re-runs
them out to 2050 with full time series. It reuses the `base_params` saved next to
those pairwise outcomes, so nothing is re-read from `constants/`.

Requires Figure 4's folder in `RUNS["policy_pairs"]`.

Slurm: `package/analysis/submit_low_policy_intensity_gen.slurm` (64 cores, 32 GB,
1 h cap; the folder is passed as `--export=ALL,PAIRWISE_FOLDERS=...`).

In [ ]:
if guard("64 calibration runs + ~13 scenarios x 64 seeds at duration_future=312"):
    pair_folder = bf.RUNS["policy_pairs"]
    if not pair_folder:
        raise RuntimeError("Figure 5 needs Figure 4's folder in RUNS['policy_pairs'] first")
    run_module("package.analysis.low_policy_intensity_gen", pair_folder)
    bf.RUNS["low_intensity"] = newest("pair_low_intensity_policies")

using("low_intensity")

In [ ]:
plot("5")

## Figure 6: histograms of consumer preference parameters

One full-length run with `save_timeseries_data_state=1`, saving the whole
controller. Its parameters are the `base_params` dict in
`package/generating_data/single_experiment_gen.py`'s `__main__` block, which is why
it runs as a module. That block also re-runs the full single-experiment plotting
set afterwards, so the run cell takes noticeably longer than the simulation itself.

No slurm job: this is a single simulation and runs locally in minutes.

In [ ]:
if guard("1 run, ~456 steps, plus the full single_experiment_plot set"):
    run_module("package.generating_data.single_experiment_gen")
    bf.RUNS["single_experiment"] = newest("single_experiment")

using("single_experiment")

In [ ]:
plot("6")

## Figure 7: EV uptake against used car market capacity

Sweeps `parameters_second_hand.max_num_cars_prop` over `[0, 0.1, 0.2, 0.5, 2]`
(`package/constants/vary_single_max_num_cars_prop.json`) against
`base_params_vary_single.json`.

Memory-bound, not CPU-bound: at `max_num_cars_prop=2` the second-hand fleet is
6,000 cars per run. The slurm job
(`package/generating_data/submit_sen_vary_max_num_cars_prop_gen.slurm`) deliberately
uses 16 cores with 128 GB rather than 64 cores, for 8 GB per worker. Locally, watch
memory before raising the worker count.

In [ ]:
if guard("5 values x 64 seeds = 320 runs at 600 steps, ~8 GB per concurrent worker"):
    run_module("package.generating_data.sen_vary_single_param_gen_second_hand_cars")
    bf.RUNS["used_car_capacity"] = newest("sen_vary_max_num_cars_prop")

using("used_car_capacity")

In [ ]:
plot("7")

---
# Supplementary figures

## Supplementary Figure 1: NPE posterior for a_chi and b_chi

Plotting only re-reads `Data/samples` and `Data/var_dict` from an existing
inference folder, which is what `RUNS["posterior"]` points at. Generation is a
separate long job: `package/calibration/sbi_single_seed_gen.py`
(`submit_sbi_single_seed.slurm`, 128 cores, 12 h cap). Re-run it only for a fresh
posterior.

The run cell below resolves `results/sbi_single_seed_*`. If you would rather plot an
existing folder, set `bf.RUNS["posterior"]` by hand and skip it.

In [ ]:
if guard("NPE calibration: thousands of ABM draws per round. Cluster job, ~12 h on 128 cores."):
    run_module("package.calibration.sbi_single_seed_gen")
    bf.RUNS["posterior"] = newest("sbi_single_seed")

using("posterior")

In [ ]:
plot("S1")

## Supplementary Figures 2-4: external data and the NK landscape

Not model output. Source images go in `docs/paper/static_figs/` under the names in
`build_figures.STATIC_SOURCES`:

| Figure | file |
|---|---|
| S2 | `electricity_vs_gasoline_prices.png` |
| S3 | `electricity_vs_gasoline_emissions.png` |
| S4 | `insights_on_NK.png` |

In [ ]:
plot("S2", "S3", "S4")

## Supplementary Figure 5: simulated against real ICE and EV price and range

Same calibration run as Figure 2, different artifact
(`Plots/multi_seed_2d_scatter.png`). If you ran Figure 2's run cell, do not run this
one: it would generate a second, identical run.

In [ ]:
if guard("Same 64 runs as Figure 2. Skip this if Figure 2's run cell already ran."):
    from package.supplementary_runs.fig05_calibration_cars_gen import run_fig5

    bf.RUNS["calibration"] = run_fig5()

using("calibration")

In [ ]:
plot("S5")

## Supplementary Figure 6: 10-panel local sensitivity of EV uptake

Ten `vary_single_param_gen` sweeps, in the paper's panel order a-j (alpha, r, mu,
kappa, b_chi, a_chi, lambda, delta, K_EV, K_ICE), combined into one figure.

Every sweep lands in a `results/single_param_vary_<timestamp>` folder whose name
says nothing about the parameter, so `package/supplementary_runs/fig06_panels.py`
identifies them by reading each folder's `Data/vary_single.pkl`. That makes the
run resumable: panels that already exist are reused, only the missing ones are
run, and one failing sweep no longer costs the other nine. The cell below prints
which panels are on disk whether or not it generates anything.

`RUNS["local_sensitivity"]` is only set once all ten exist -- a partial list
would plot a figure with blank panels.

Slurm: `package/supplementary_runs/submit_fig06_panels.slurm` (2,752 runs, about
13 min at 64 cores; re-submit it after a failure and it picks up where it
stopped).

In [ ]:
from package.supplementary_runs import fig06_panels

if guard("10 params x 4 values (b_chi has 7) x 64 seeds = 2,752 runs (~13 min on 64 cores)"):
    # Reuses any panel that already has a run; only the missing ones simulate.
    combined_folder, failures = fig06_panels.run_all()
    print("Combined output:", combined_folder)

# Panel status either way, so a partial set is visible without generating.
resolved, _ = fig06_panels.resolve_panels()
if all(folder for _, folder in resolved):
    bf.RUNS["local_sensitivity"] = [folder for _, folder in resolved]

using("local_sensitivity")

In [ ]:
plot("S6")

## Supplementary Figures 7 and 8: Sobol first-order and total-order indices

Saltelli sampling over the same ten parameters as Figure S6, six outputs. One run
makes both figures. `N_samples` is baked into the artifact filename, so the value in
`fig07_08_sobol_gen.N_SAMPLES` and the value in `build_figures.SOBOL_N_SAMPLES` must
agree; the run cell checks this.

**This is the most expensive job in the paper**: 196,608 runs, about 1,092 core-hours,
roughly 8.5 h on 128 cores
(`package/supplementary_runs/submit_fig07_08_sobol_gen.slurm`). Do not run it locally.

In [ ]:
if guard("196,608 runs, ~1,092 core-hours (~8.5 h on 128 cores). Cluster only."):
    from package.supplementary_runs.fig07_08_sobol_gen import N_SAMPLES, run_fig7_8

    if N_SAMPLES != bf.SOBOL_N_SAMPLES:
        raise RuntimeError(
            f"N_SAMPLES={N_SAMPLES} but build_figures.SOBOL_N_SAMPLES={bf.SOBOL_N_SAMPLES}; "
            "the artifact filename is keyed on it, so make them match"
        )
    bf.RUNS["sobol"] = run_fig7_8()

using("sobol")

In [ ]:
plot("S7", "S8")

## Supplementary Figures 9 and 10: BAU sensitivity to grid intensity and electricity price

One generation run over 4 grid intensities x 3 electricity prices x 16 seeds serves
both figures: Figure 9 shows three reduction columns, Figure 10 needs the extra
"no change" value for its elasticity baseline.

Slurm: `package/supplementary_runs/submit_fig09_10_bau_gen.slurm` (192 runs, ~4 min).
Small enough to run locally.

In [ ]:
if guard("4 grid intensities x 3 electricity prices x 16 seeds = 192 runs (~4 min on 64 cores)"):
    from package.supplementary_runs.fig09_10_bau_gen import run_fig9_10

    bf.RUNS["bau_grid"] = run_fig9_10()

using("bau_grid")

In [ ]:
plot("S9", "S10")

## Supplementary Figures 11-14: EV uptake policy grids

Four independent runs of `fig11_14_policy_grid_gen.py`, one per figure:

| Figure | grid | `RUNS` key |
|---|---|---|
| S11 | beta multiplier x carbon price | `grid_beta_carbon` |
| S12 | beta multiplier x new car rebate | `grid_beta_rebate` |
| S13 | a_chi x carbon price | `grid_achi_carbon` |
| S14 | a_chi x new car rebate | `grid_achi_rebate` |

Each is 3,584 runs, about 19 min at 64 cores
(`submit_fig11_beta_carbon_gen.slurm` and its three siblings).

**The BAU baseline.** The heatmap overlay needs `data_cross_bau.pkl`, and a `cross_*`
folder only has it if its own BAU sweep finished. The BAU baseline contains no
policy, so it depends only on the physical parameter: one BAU run serves both
policies on that axis. `RUNS["grid_bau_beta"]` and `RUNS["grid_bau_achi"]` name the
folder to borrow it from. Leave one empty to use each figure's own folder.

In the `RUNS` shipped in `build_figures.py`, only the beta axis has that file:
`grid_bau_beta` points at the beta/rebate folder, and `grid_bau_achi` is empty
because neither `cross_a_chi_*` folder holds `data_cross_bau.pkl`. So S11 and S12
plot as they are, and S13 and S14 report a missing `data_cross_bau` until an a_chi
BAU sweep finishes. Point `RUNS["grid_bau_achi"]` at whichever a_chi folder ends up
with that file.

In [ ]:
FIGURE_TO_RUN_KEY = {
    "11": "grid_beta_carbon",
    "12": "grid_beta_rebate",
    "13": "grid_achi_carbon",
    "14": "grid_achi_rebate",
}

# Narrow this to regenerate only some of the four.
GRID_FIGURES = ["11", "12", "13", "14"]

if guard(f"{len(GRID_FIGURES)} x 3,584 runs, ~19 min each on 64 cores"):
    from package.supplementary_runs.fig11_14_policy_grid_gen import run_policy_grid_figure

    for number in GRID_FIGURES:
        bf.RUNS[FIGURE_TO_RUN_KEY[number]] = run_policy_grid_figure(number)

for number in ["11", "12", "13", "14"]:
    using(FIGURE_TO_RUN_KEY[number])
using("grid_bau_beta")
using("grid_bau_achi")

In [ ]:
plot("S11", "S12", "S13", "S14")

---
## Everything at once

Re-plots every figure that has data. Cheap for most figures, but supplementary
Figure 6's combined panel and the Sobol figures re-run their analysis step, so give
it a few minutes.

In [ ]:
for key in FIG:
    plot(key)

## Stage the figures into docs/paper and rewrite the .tex

The notebook never touches the `.tex`. Do that from the command line, where the
figure numbering, the backups (`<name>.tex.bak`) and the skip logic all live:

```bash
python -m package.paper_figures.build_figures --list          # what is ready
python -m package.paper_figures.build_figures                 # build everything, rewrite the .tex
python -m package.paper_figures.build_figures --only 3,S6     # a subset
python -m package.paper_figures.build_figures --skip-plot     # copy the existing PNGs only
python -m package.paper_figures.build_figures --no-tex        # copy, leave the .tex alone
```

Folders you generated in this session live in `bf.RUNS` for the life of the kernel
only. To keep them, paste them into the `RUNS` dict at the top of
`package/paper_figures/build_figures.py`. The cell below prints them in that form.

In [ ]:
print("RUNS = {")
for key, value in bf.RUNS.items():
    if isinstance(value, list):
        print(f'    "{key}": [')
        for folder in value:
            print(f'        "{folder}",')
        print("    ],")
    else:
        print(f'    "{key}": "{value}",')
print("}")